# V7 AHB Validation: Style-Controlled Compassion Probes

This notebook trains linear probes on **v7 style-controlled pairs** and validates against AHB-graded outputs.

**Key difference from v5:**
- **V5 pairs**: Style-confounded (compassionate = informal, non-compassionate = formal)
- **V7 pairs**: Style-controlled (only framing differs - welfare vs economic focus)

**Previous result (v5):** r = 0.457

If v7 achieves similar or higher correlation, this suggests the probe captures genuine compassion rather than style artifacts.

## Cell 1: Setup & Dependencies

In [ ]:
# Install uv for faster package installation
!pip install -q uv

# Install dependencies with uv (much faster than pip)
!uv pip install --system -q transformers accelerate torch scikit-learn scipy tqdm

# Note: flash-attn is skipped - it requires building from source on Colab
# which typically gets OOM-killed. The model will use standard attention instead.
print("Dependencies installed (using standard attention, not flash-attn)")

## Cell 1b: Hugging Face Authentication

Llama is a gated model. You need to:
1. Accept the license at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
2. Create an access token at https://huggingface.co/settings/tokens
3. Run the cell below and paste your token

In [ ]:
from huggingface_hub import login

# This will prompt for your token (or use Colab secrets)
# You can also set: login(token="hf_xxx")
login()

## Cell 2: Imports & Configuration

In [ ]:
import torch
import numpy as np
import json
from pathlib import Path
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score, accuracy_score
from scipy.stats import pearsonr, spearmanr

# Configuration
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
LAYERS = [8, 12, 16, 20, 24, 28]  # Layers to probe

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Cell 3: Load Model

In [ ]:
print(f"Loading model: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Try Flash Attention first, fall back to standard
try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        attn_implementation="flash_attention_2"
    )
    print("Using Flash Attention 2")
except Exception as e:
    print(f"Flash Attention not available: {e}")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )
    print("Using standard attention")

model.eval()

# Report memory usage
if torch.cuda.is_available():
    mem_used = torch.cuda.memory_allocated() / 1e9
    mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU memory: {mem_used:.1f} / {mem_total:.1f} GB")

## Cell 4: Load Data

Upload these files to Colab (drag into the folder icon in left sidebar):
- `pairs_v7_full.jsonl` — v7 style-controlled contrastive pairs
- `llama_8b_graded.jsonl` — AHB-graded model outputs for validation

In [ ]:
import json

def load_jsonl(path):
    """Load JSONL file."""
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

# Files uploaded directly to Colab - no GitHub API needed
# Upload these files via the folder icon in the left sidebar:
#   - pairs_v7_full.jsonl
#   - llama_8b_graded.jsonl

v7_pairs = load_jsonl("pairs_v7_full.jsonl")
graded_outputs = load_jsonl("llama_8b_graded.jsonl")

print(f"Loaded {len(v7_pairs)} v7 style-controlled pairs")
print(f"Loaded {len(graded_outputs)} AHB-graded outputs")

# Preview a pair
print("\nSample v7 pair:")
print(f"  Question: {v7_pairs[0].get('question', v7_pairs[0].get('prompt', ''))[:100]}...")

## Cell 5: Extraction Functions

In [ ]:
class ActivationCapture:
    """Hook-based activation capture for memory efficiency."""

    def __init__(self, layer_idx: int):
        self.layer_idx = layer_idx
        self.activation = None
        self.hook = None

    def hook_fn(self, module, input, output):
        hidden = output[0] if isinstance(output, tuple) else output
        self.activation = hidden.detach()

    def register(self, model):
        layer = model.model.layers[self.layer_idx]
        self.hook = layer.register_forward_hook(self.hook_fn)

    def remove(self):
        if self.hook:
            self.hook.remove()
            self.hook = None

    def get_activation(self) -> torch.Tensor:
        return self.activation


def format_conversation(pair: dict, response_type: str, tokenizer) -> tuple:
    """
    Format a conversation for the model and compute response start index.

    Returns:
        tuple: (formatted_text, response_start_idx)
    """
    # Support both 'prompt' and 'question' keys
    user_content = pair.get("prompt") or pair.get("question")

    # Get prompt-only to find where response starts
    prompt_only = [{"role": "user", "content": user_content}]
    prompt_text = tokenizer.apply_chat_template(prompt_only, tokenize=False, add_generation_prompt=True)
    prompt_tokens = tokenizer(prompt_text, truncation=True, max_length=1024)
    response_start_idx = len(prompt_tokens["input_ids"])

    # Get full conversation
    conversation = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": pair[response_type]}
    ]
    full_text = tokenizer.apply_chat_template(conversation, tokenize=False)

    return full_text, response_start_idx


def extract_activation_with_hook(
    model,
    tokenizer,
    text: str,
    layer: int,
    response_start_idx: int = 0
) -> torch.Tensor:
    """
    Extract mean-pooled activation using hooks (memory efficient).
    Only pools over response tokens, not the prompt.
    """
    tokens = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024)
    tokens = {k: v.to(model.device) for k, v in tokens.items()}

    capture = ActivationCapture(layer)
    capture.register(model)

    try:
        with torch.no_grad():
            model(**tokens)

        activation = capture.get_activation()  # (1, seq, hidden)
        response_acts = activation[0, response_start_idx:, :]

        return response_acts.mean(dim=0).cpu()
    finally:
        capture.remove()


def extract_single_layer(
    model,
    tokenizer,
    pairs: list,
    layer: int,
    batch_size: int = 8,
    show_progress: bool = True
) -> dict:
    """
    Extract activations for a single layer across all pairs.

    Returns:
        dict with 'compassionate' and 'non_compassionate' tensors
    """
    compassionate_acts = []
    non_compassionate_acts = []

    iterator = tqdm(pairs, desc=f"Layer {layer}", disable=not show_progress)

    for i, pair in enumerate(iterator):
        # Extract compassionate response activation
        text_comp, response_start_comp = format_conversation(pair, "compassionate_response", tokenizer)
        act_comp = extract_activation_with_hook(model, tokenizer, text_comp, layer, response_start_idx=response_start_comp)
        compassionate_acts.append(act_comp)

        # Extract non-compassionate response activation
        text_non, response_start_non = format_conversation(pair, "non_compassionate_response", tokenizer)
        act_non = extract_activation_with_hook(model, tokenizer, text_non, layer, response_start_idx=response_start_non)
        non_compassionate_acts.append(act_non)

        # Clear cache periodically
        if (i + 1) % batch_size == 0:
            torch.cuda.empty_cache()

    return {
        "compassionate": torch.stack(compassionate_acts),
        "non_compassionate": torch.stack(non_compassionate_acts)
    }

print("Extraction functions defined.")

## Cell 6: Extract V7 Activations

In [ ]:
print(f"Extracting activations for {len(v7_pairs)} v7 pairs across {len(LAYERS)} layers...")
print(f"Layers: {LAYERS}")

activations = {}
for layer in LAYERS:
    print(f"\nExtracting layer {layer}...")
    activations[layer] = extract_single_layer(model, tokenizer, v7_pairs, layer)
    torch.cuda.empty_cache()
    
    # Report shape
    shape = activations[layer]["compassionate"].shape
    print(f"  Shape: {shape} (n_pairs, hidden_dim)")

print("\nExtraction complete!")

## Cell 7: Training Functions

In [ ]:
def prepare_data(activations: dict, layer: int) -> tuple:
    """
    Prepare training data from activations.

    Returns:
        X: (n_samples, hidden_dim) activation vectors
        y: (n_samples,) binary labels (1=compassionate, 0=not)
    """
    layer_data = activations[layer]

    # Stack and convert bf16 -> float32 for sklearn
    comp = layer_data['compassionate'].float().numpy()
    non_comp = layer_data['non_compassionate'].float().numpy()

    X = np.vstack([comp, non_comp])
    y = np.array([1] * len(comp) + [0] * len(non_comp))

    return X, y


def train_probe(X: np.ndarray, y: np.ndarray) -> tuple:
    """
    Train logistic regression probe.

    Returns:
        direction: normalized weight vector
        probe: trained sklearn model
        metrics: dict of evaluation metrics
    """
    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Train probe with cross-validation for regularization
    probe = LogisticRegressionCV(
        Cs=10,
        cv=5,
        max_iter=1000,
        random_state=42
    )
    probe.fit(X_train, y_train)

    # Extract direction (normalized weights)
    direction = probe.coef_[0]
    direction = direction / np.linalg.norm(direction)

    # Compute metrics
    y_pred = probe.predict(X_test)
    y_prob = probe.predict_proba(X_test)[:, 1]

    cv_scores = cross_val_score(probe, X, y, cv=5)

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "auroc": roc_auc_score(y_test, y_prob),
        "cv_accuracy_mean": cv_scores.mean(),
        "cv_accuracy_std": cv_scores.std(),
        "n_train": len(X_train),
        "n_test": len(X_test),
        "best_C": probe.C_[0],
    }

    # Random label control (sanity check)
    y_shuffled = np.random.permutation(y_train)
    probe_random = LogisticRegressionCV(cv=5, max_iter=1000, random_state=42)
    probe_random.fit(X_train, y_shuffled)
    metrics["random_label_accuracy"] = probe_random.score(X_test, y_test)

    return direction, probe, metrics

print("Training functions defined.")

## Cell 8: Train V7 Probes

In [ ]:
print("Training probes on v7 style-controlled pairs...\n")

probes = {}
for layer in LAYERS:
    print(f"Layer {layer}:")
    X, y = prepare_data(activations, layer)
    direction, probe, metrics = train_probe(X, y)
    probes[layer] = {
        "direction_probe": direction,
        "probe_model": probe,
        "metrics": metrics
    }
    print(f"  Accuracy: {metrics['accuracy']:.1%}")
    print(f"  AUROC: {metrics['auroc']:.3f}")
    print(f"  CV: {metrics['cv_accuracy_mean']:.1%} +/- {metrics['cv_accuracy_std']:.1%}")
    print(f"  Random control: {metrics['random_label_accuracy']:.1%} (should be ~50%)")
    print()

# Find best layer
best_layer = max(probes.keys(), key=lambda l: probes[l]["metrics"]["accuracy"])
print(f"Best layer: {best_layer} (accuracy: {probes[best_layer]['metrics']['accuracy']:.1%})")

## Cell 9: Validation Functions

In [ ]:
def compute_response_start_idx(prompt: str, tokenizer) -> int:
    """
    Compute the token index where the assistant response begins.
    This matches the logic from extract.py's format_conversation().
    """
    prompt_only = [{"role": "user", "content": prompt}]
    prompt_text = tokenizer.apply_chat_template(prompt_only, tokenize=False, add_generation_prompt=True)
    prompt_tokens = tokenizer(prompt_text, truncation=True, max_length=1024)
    return len(prompt_tokens["input_ids"])


def extract_and_project(
    model,
    tokenizer,
    prompt: str,
    response: str,
    direction: np.ndarray,
    layer: int,
) -> float:
    """
    Extract activation and project onto compassion direction.
    Uses the same response token extraction logic as training.
    """
    response_start_idx = compute_response_start_idx(prompt, tokenizer)

    # Format full conversation
    conversation = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": response}
    ]
    formatted = tokenizer.apply_chat_template(conversation, tokenize=False)
    tokens = tokenizer(formatted, return_tensors="pt", truncation=True, max_length=1024)
    tokens = {k: v.to(model.device) for k, v in tokens.items()}

    # Extract activations using hooks
    capture = ActivationCapture(layer)
    capture.register(model)

    try:
        with torch.no_grad():
            model(**tokens)
        activations = capture.get_activation()
    finally:
        capture.remove()

    # Mean-pool over response tokens only
    seq_len = activations.shape[1]

    # Handle edge case where response_start_idx exceeds sequence length
    if response_start_idx >= seq_len:
        response_start_idx = max(0, seq_len - 10)

    response_acts = activations[0, response_start_idx:, :].mean(dim=0).float().cpu().numpy()

    # Project onto direction
    projection = np.dot(response_acts, direction)
    return float(projection)


def compute_correlations(probe_scores: list, ahb_scores: list) -> dict:
    """Compute Pearson and Spearman correlations with p-values."""
    if len(probe_scores) < 3:
        return {
            "pearson_r": None,
            "pearson_p": None,
            "spearman_r": None,
            "spearman_p": None,
            "n": len(probe_scores)
        }

    pearson_r, pearson_p = pearsonr(probe_scores, ahb_scores)
    spearman_r, spearman_p = spearmanr(probe_scores, ahb_scores)

    return {
        "pearson_r": float(pearson_r),
        "pearson_p": float(pearson_p),
        "spearman_r": float(spearman_r),
        "spearman_p": float(spearman_p),
        "n": len(probe_scores)
    }

print("Validation functions defined.")

## Cell 10: Run AHB Validation

In [ ]:
print(f"Running AHB validation using layer {best_layer} probe...")
print(f"Processing {len(graded_outputs)} graded outputs...\n")

direction = probes[best_layer]["direction_probe"]

probe_scores = []
ahb_scores = []
per_dimension_data = {}

for i, output in enumerate(tqdm(graded_outputs, desc="Validating")):
    question = output["question"]
    response = output["response"]
    dimension_scores = output.get("dimension_scores", {})
    overall_score = output.get("overall_score", 0.0)

    # Skip if no dimension scores
    if not dimension_scores:
        continue

    # Compute probe score
    try:
        probe_score = extract_and_project(
            model, tokenizer, question, response, direction, best_layer
        )
    except Exception as e:
        print(f"Warning: Failed to process output {output.get('id', i)}: {e}")
        continue

    probe_scores.append(probe_score)
    ahb_scores.append(overall_score)

    # Track per-dimension data
    for dim, score in dimension_scores.items():
        if dim not in per_dimension_data:
            per_dimension_data[dim] = {"probe_scores": [], "ahb_scores": []}
        per_dimension_data[dim]["probe_scores"].append(probe_score)
        per_dimension_data[dim]["ahb_scores"].append(score)

    # Clear CUDA cache periodically
    if (i + 1) % 20 == 0:
        torch.cuda.empty_cache()

print(f"\nProcessed {len(probe_scores)} outputs successfully.")

## Cell 11: Compute Correlations & Results

In [ ]:
print("=" * 60)
print("V7 PROBE VALIDATION RESULTS")
print("=" * 60)

# Overall correlation
overall_corr = compute_correlations(probe_scores, ahb_scores)
print(f"\nOverall AHB Score Correlation (n={overall_corr['n']})")
print(f"  Pearson r:  {overall_corr['pearson_r']:.3f} (p={overall_corr['pearson_p']:.4f})")
print(f"  Spearman r: {overall_corr['spearman_r']:.3f} (p={overall_corr['spearman_p']:.4f})")

# Per-dimension correlations
print("\nPer-Dimension Correlations:")
per_dimension_correlations = {}

for dim in sorted(per_dimension_data.keys()):
    data = per_dimension_data[dim]
    corr = compute_correlations(data["probe_scores"], data["ahb_scores"])
    per_dimension_correlations[dim] = corr

    if corr["pearson_r"] is not None:
        sig = "*" if corr["pearson_p"] < 0.05 else ""
        print(f"  {dim}: r={corr['pearson_r']:.3f}{sig} (n={corr['n']})")

# Probe statistics
print(f"\nProbe Score Statistics:")
print(f"  Mean: {np.mean(probe_scores):.3f}")
print(f"  Std:  {np.std(probe_scores):.3f}")
print(f"  Min:  {np.min(probe_scores):.3f}")
print(f"  Max:  {np.max(probe_scores):.3f}")

## Cell 12: Compare V5 vs V7 Results

In [ ]:
# V5 results (from previous run with style-confounded pairs)
V5_PEARSON = 0.457
V5_SPEARMAN = 0.443  # approximate, update if known

v7_pearson = overall_corr['pearson_r']
v7_spearman = overall_corr['spearman_r']

print("=" * 60)
print("V5 vs V7 COMPARISON")
print("=" * 60)
print()
print(f"{'Metric':<20} {'V5 (confounded)':<20} {'V7 (controlled)':<20} {'Difference':<15}")
print("-" * 75)
print(f"{'Pearson r':<20} {V5_PEARSON:<20.3f} {v7_pearson:<20.3f} {v7_pearson - V5_PEARSON:+.3f}")
print(f"{'Spearman r':<20} {V5_SPEARMAN:<20.3f} {v7_spearman:<20.3f} {v7_spearman - V5_SPEARMAN:+.3f}")
print()

# Interpretation
print("\nINTERPRETATION:")
print("-" * 60)

diff = v7_pearson - V5_PEARSON
if diff > 0.05:
    interpretation = (
        "V7 (style-controlled) OUTPERFORMS V5 (style-confounded).\n"
        "This strongly suggests the probe captures genuine compassion,\n"
        "not just style artifacts. The style confound in V5 may have\n"
        "actually been hurting correlation."
    )
elif diff > -0.05:
    interpretation = (
        "V7 and V5 achieve SIMILAR correlation.\n"
        "This suggests the probe captures genuine compassion signal,\n"
        "as removing style confounds didn't reduce correlation.\n"
        "The v5 correlation was NOT primarily driven by style."
    )
else:
    interpretation = (
        "V7 (style-controlled) shows LOWER correlation than V5.\n"
        "This suggests the v5 probe may have been partially measuring\n"
        "style artifacts. The v7 probe is a more conservative measure\n"
        "of genuine compassion signal."
    )

print(interpretation)

# Overall assessment
print("\n" + "=" * 60)
if v7_pearson > 0.3:
    print("CONCLUSION: Moderate-to-strong evidence that probe captures")
    print("genuine compassion signal, validated by style-controlled pairs.")
elif v7_pearson > 0.15:
    print("CONCLUSION: Weak-to-moderate evidence. Signal exists but")
    print("may benefit from larger/better training data.")
else:
    print("CONCLUSION: Weak evidence. Probe may need refinement.")
print("=" * 60)

## Cell 13: Save Results (Optional)

In [ ]:
# Save results to JSON
results = {
    "config": {
        "model": MODEL_NAME,
        "pairs_version": "v7",
        "n_pairs": len(v7_pairs),
        "n_graded_outputs": len(graded_outputs),
        "best_layer": best_layer,
        "layers_evaluated": LAYERS
    },
    "probe_metrics": {
        str(layer): probes[layer]["metrics"] for layer in LAYERS
    },
    "validation": {
        "overall_correlation": overall_corr,
        "per_dimension_correlations": per_dimension_correlations,
        "probe_statistics": {
            "mean": float(np.mean(probe_scores)),
            "std": float(np.std(probe_scores)),
            "min": float(np.min(probe_scores)),
            "max": float(np.max(probe_scores))
        }
    },
    "comparison": {
        "v5_pearson": V5_PEARSON,
        "v7_pearson": v7_pearson,
        "difference": v7_pearson - V5_PEARSON
    }
}

with open("v7_ahb_validation_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Results saved to v7_ahb_validation_results.json")

# Also save the probe direction for later use
torch.save({
    "direction": probes[best_layer]["direction_probe"],
    "layer": best_layer,
    "metrics": probes[best_layer]["metrics"],
    "pairs_version": "v7"
}, "v7_compassion_probe.pt")

print("Probe saved to v7_compassion_probe.pt")

## Cell 14: Visualization (Optional)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: Scatter of probe scores vs AHB scores
ax1 = axes[0]
ax1.scatter(probe_scores, ahb_scores, alpha=0.5, s=30)
ax1.set_xlabel("Probe Score (V7)")
ax1.set_ylabel("AHB Overall Score")
ax1.set_title(f"V7 Probe vs AHB Correlation (r={v7_pearson:.3f})")

# Add trend line
z = np.polyfit(probe_scores, ahb_scores, 1)
p = np.poly1d(z)
x_line = np.linspace(min(probe_scores), max(probe_scores), 100)
ax1.plot(x_line, p(x_line), "r--", alpha=0.8, label="Trend")
ax1.legend()

# Plot 2: Layer comparison
ax2 = axes[1]
layers_sorted = sorted(LAYERS)
accuracies = [probes[l]["metrics"]["accuracy"] for l in layers_sorted]
ax2.bar(range(len(layers_sorted)), accuracies, tick_label=[str(l) for l in layers_sorted])
ax2.set_xlabel("Layer")
ax2.set_ylabel("Probe Accuracy")
ax2.set_title("Probe Accuracy by Layer (V7 Pairs)")
ax2.axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label="Chance")
ax2.legend()
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.savefig("v7_validation_plots.png", dpi=150)
plt.show()

print("Plots saved to v7_validation_plots.png")